# Loom — kernel for multi-agent LLM rooms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hdubey-debug/loom/blob/main/examples/colab_demo.ipynb)

Loom is a small Python kernel that turns ordinary callables (any LLM, any wrapper, any scripted function) into *agents* sharing a single message bus. The kernel handles event ordering, lease arbitration, obligation tracking, dead-letter routing, and a journal. **You** drop in a `ConversationPolicy` to decide *who* may speak, *when*, and *what extra instructions* to render.

This notebook has two halves:

1. **Mock-agent demos (no API keys).** Walk through the four bundled policies plus a custom subclass that uses a v0.2 hook.
2. **Real-LLM live chat.** Plug in your OpenAI / Gemini keys, wire up three real agents, and have a back-and-forth conversation in the notebook. Swap policies by editing one line and re-running the cell.

**Cells run top to bottom.** Mock half is ~30s (mostly the install). Real half costs whatever you'd pay for the calls (a few cents at most with default models).

---

Repo: https://github.com/hdubey-debug/loom

## 0. Install Loom

Loom is pure Python; no GPU needed. Install takes ~30 seconds in Colab.

In [ ]:
!pip install -q git+https://github.com/hdubey-debug/loom.git

## 1. Setup — three mock agents + a result printer

Real Loom agents are anything matching the `Agent` protocol — `id` plus a `stream(prompt) -> Iterator[str]`. The bundled `agent_from_send` adapter wraps a one-shot `prompt -> str` callable. Each mock agent below has a fixed personality so you can see, demo by demo, *which* agents the policy let speak.

In [ ]:
from loom import LoomRoom, agent_from_send

def alice_send(prompt: str) -> str:
    return "Alice: I'd start by clarifying what success looks like."

def bob_send(prompt: str) -> str:
    return "Bob: Counterpoint — test the binding constraint before planning."

def carol_send(prompt: str) -> str:
    return "Carol: Synthesis — name the strongest version of each side, then choose."

alice = agent_from_send("alice", alice_send, persona="planner")
bob   = agent_from_send("bob",   bob_send,   persona="critic")
carol = agent_from_send("carol", carol_send, persona="synthesiser")

def show(result, label=""):
    """Pretty-print a TurnResult."""
    print(f"=== {label} ===")
    print(f"  routing      : {result.routing_case}")
    print(f"  closed_reason: {result.closed_reason}")
    print(f"  replies      : {len(result.messages)}")
    for m in result.messages:
        print(f"    [{m.sender}] {m.body}")
    print()

## 2. `OpenChatPolicy` — broadcast to everyone

Every active capable participant gets to respond. With three agents and one user post, expect three replies in arrival order.

Note the `with room:` context manager — actor threads start on `__enter__` and stop on `__exit__`.

In [ ]:
from loom import OpenChatPolicy

with LoomRoom(agents=[alice, bob, carol], policy=OpenChatPolicy()) as room:
    result = room.post_and_wait("Should we ship the new feature this week?")
    show(result, "OpenChatPolicy — all three should respond")

## 3. `SingleResponderPolicy` — route everything to one agent

Use this for a chatbot / single-model copilot shape: the policy always routes to one configured pid. The other two agents see the user post but the kernel rejects their lease attempts.

In [ ]:
from loom import SingleResponderPolicy

with LoomRoom(agents=[alice, bob, carol],
              policy=SingleResponderPolicy("bob")) as room:
    result = room.post_and_wait("Should we ship the new feature this week?")
    show(result, "SingleResponderPolicy(bob) — only Bob should respond")

## 4. `RoundRobinPolicy` — strict rotation

Each user post advances the rotation pointer by exactly one speaker. Three posts cycle through `[alice, bob, carol]`.

In [ ]:
from loom import RoundRobinPolicy

with LoomRoom(agents=[alice, bob, carol],
              policy=RoundRobinPolicy(["alice", "bob", "carol"])) as room:
    for i in range(3):
        result = room.post_and_wait(f"Round {i+1}: anything to add?")
        show(result, f"RoundRobinPolicy — turn {i+1}")

## 5. `DefaultPolicy` — `@-mention` for direct routing

`DefaultPolicy` is the production-grade one: vocative detection, fallback chains, and direct-mention priority. An `@<id>` in the user's message routes that turn straight to that participant.

In [ ]:
from loom import DefaultPolicy

with LoomRoom(agents=[alice, bob, carol], policy=DefaultPolicy()) as room:
    show(room.post_and_wait("@alice what's your read on this?"),
         "DefaultPolicy + @alice")
    show(room.post_and_wait("@carol you wrap us up."),
         "DefaultPolicy + @carol")

## 6. Custom policy — using a v0.2 hook

v0.2 added several optional hooks on `ConversationPolicy`. The simplest one to demo is `charter_text` — a string the kernel injects into every prompt's preamble immediately after the fixed `LOOM_PROTOCOL_INSTRUCTIONS` charter.

Here we subclass `OpenChatPolicy` and add a brevity rule. With a real LLM you'd see all three agents follow it.

Other v0.2 hooks you can override the same way: `dead_letter_target`, `should_post_response`, `prompt_sections`. Plus pluggable `RoomConfig.lease_checks` and `RoomConfig.trigger_priority`.

In [ ]:
class StrictBrevityPolicy(OpenChatPolicy):
    """OpenChat plus a one-line charter rule."""

    def charter_text(self, state):
        return ("BREVITY RULE: every reply must be a single sentence, "
                "max 15 words. No bullet lists.")

with LoomRoom(agents=[alice, bob, carol],
              policy=StrictBrevityPolicy()) as room:
    result = room.post_and_wait("How do we ship faster?")
    show(result, "Custom StrictBrevityPolicy — charter rule injected")

## 7. Bonus — peek at the bus

Loom journals every event into an append-only log. Below we reach into the underlying session (the `_session` attr is intentionally semi-private — for inspection only) and dump every event posted during one turn.

You'll see the user post, control events (`user_turn_opened`, `obligation_recorded`, lease grants), stream events, the committed `chat` events, and the closing control event. This is exactly what the journal would write to disk for replay.

In [ ]:
with LoomRoom(agents=[alice, bob, carol], policy=OpenChatPolicy()) as room:
    result = room.post_and_wait("ping")
    bus = room._session.bus
    print(f"bus length: {len(bus)} events\n")
    for ev in bus.snapshot():
        body = str(ev.body)
        if len(body) > 70:
            body = body[:67] + "..."
        print(f"  id={ev.id:3d}  kind={ev.kind:7s}  sender={ev.sender:7s}  {body}")

---

# Part 2 — Real LLM agents + live chat

Plug in your own keys, wire three real agents (two GPT, one Gemini), then have a back-and-forth conversation in this notebook. You can swap the policy mid-notebook by editing one line and re-running the chat cell.

Skip this half if you don't have keys handy — the mock demos above already cover the API surface.

## 8. Install provider SDKs

Two SDKs — `openai` for GPT-class models, `google-generativeai` for Gemini. ~10 seconds in Colab.

In [ ]:
!pip install -q openai google-generativeai

## 9. Add your API keys

Keys are read from environment variables (`OPENAI_API_KEY`, `GEMINI_API_KEY`). The cell below uses `getpass` so the key never appears in cell output, and re-uses an env var if one is already set (e.g. via Colab's **🔑 Secrets** panel).

**Where to get keys:**

- OpenAI: https://platform.openai.com/api-keys
- Gemini: https://aistudio.google.com/app/apikey (free tier available)

If you only have one of them, leave the other blank — you can drop the missing-provider agent from the wiring cell below.

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (paste; hidden, blank to skip): ")
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY (paste; hidden, blank to skip): ")

have_openai = bool(os.environ.get("OPENAI_API_KEY"))
have_gemini = bool(os.environ.get("GEMINI_API_KEY"))
print(f"OpenAI configured: {have_openai}   |   Gemini configured: {have_gemini}")

## 10. Wire up real agents

Two helpers — `make_openai_agent` and `make_gemini_agent` — each take an agent id, a model, and a system prompt. The model is selected per-agent here; you can mix any models you have access to.

Defaults are cheap-and-fast: `gpt-4o-mini` for OpenAI, `gemini-1.5-flash` for Gemini. Override the `model=` argument to use anything else (`gpt-4o`, `gpt-4.1`, `o3-mini`, `gemini-1.5-pro`, `gemini-2.0-flash`, etc.).

In [ ]:
def make_openai_agent(agent_id, system, *, model="gpt-4o-mini", persona=""):
    from openai import OpenAI
    client = OpenAI()  # honours OPENAI_API_KEY env var

    def send(prompt: str) -> str:
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
        )
        return r.choices[0].message.content or ""

    return agent_from_send(agent_id, send, persona=persona)


def make_gemini_agent(agent_id, system, *, model="gemini-1.5-flash", persona=""):
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    m = genai.GenerativeModel(model_name=model, system_instruction=system)

    def send(prompt: str) -> str:
        r = m.generate_content(prompt)
        return (r.text or "").strip()

    return agent_from_send(agent_id, send, persona=persona)


# Build whichever subset of agents you have keys for.
real_agents = []
if have_openai:
    real_agents.append(make_openai_agent(
        "planner",
        "You are a concise planner. Identify the goal and the next concrete step. "
        "Two short paragraphs maximum.",
        persona="planner",
    ))
    real_agents.append(make_openai_agent(
        "critic",
        "You are a concise critic. Find the strongest objection to the plan on the table. "
        "One short paragraph.",
        persona="critic",
    ))
if have_gemini:
    real_agents.append(make_gemini_agent(
        "synthesiser",
        "You are a concise synthesiser. Read what the others said and produce the best version, "
        "naming the strongest take and the binding tradeoff. One short paragraph.",
        persona="synthesiser",
    ))

print(f"agents wired: {[a.id for a in real_agents]}")
if not real_agents:
    print("⚠️  No agents — add at least one API key in the cell above.")

## 11. Live chat — pick a policy and go

Edit the `POLICY = ...` line below to change behavior, then re-run the cell to start a new chat session with the chosen policy. Type messages, press Enter, and watch the agents respond. `/quit` (or empty input) ends the session.

Quick options to swap in:

```python
POLICY = OpenChatPolicy()                              # broadcast to all
POLICY = SingleResponderPolicy("planner")              # one expert
POLICY = RoundRobinPolicy(["planner", "critic", "synthesiser"])  # strict rotation
POLICY = DefaultPolicy()                               # @-mentions for direct routing
POLICY = StrictBrevityPolicy()                         # custom: brevity rule injected
```

**Try this**: post a question normally, then post `@critic only — what could go wrong?` with `DefaultPolicy` to see direct routing fire.

In [ ]:
POLICY = OpenChatPolicy()  # ← edit me to swap policies

if not real_agents:
    print("No agents — add a key above and re-run the wiring cell.")
else:
    with LoomRoom(agents=real_agents, policy=POLICY) as room:
        print(f"Live chat — policy: {type(POLICY).__name__}")
        print(f"Agents: {[a.id for a in real_agents]}")
        print("Type a message and press Enter. Type /quit (or blank) to exit.\n")
        while True:
            try:
                user_text = input("you> ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\n(interrupted)")
                break
            if not user_text or user_text.lower() in ("/quit", "/exit", "q"):
                print("(session ended)")
                break
            try:
                result = room.post_and_wait(user_text, timeout=60)
            except Exception as exc:
                print(f"  ⚠️  error: {type(exc).__name__}: {exc}")
                continue
            print(f"  ↪ routing={result.routing_case}  closed={result.closed_reason}  "
                  f"replies={len(result.messages)}")
            for m in result.messages:
                print(f"  [{m.sender}] {m.body}\n")

## 12. Where to next

- **Mix any models you have access to**: `make_openai_agent(..., model="gpt-4o")`, `make_gemini_agent(..., model="gemini-2.0-flash")`, etc. The model is per-agent.
- **DM channels**: `room.post("sidebar message", channel="dm:planner")` — visible only to planner + user + system. Useful for clarifications without polluting the main thread.
- **Custom `LeaseCheck`**: write your own gate (e.g. "agents must hold an active certification slot") and pass it via `RoomConfig(lease_checks=(MyCheck(), *DEFAULT_LEASE_CHECKS))`.
- **Roadmap**: v0.3 adds a *controller* mechanism (one agent's posts open chained turns — the CEO/orchestrator pattern); v0.4 adds structured `tool_call` / `tool_result` events; v0.5 explores Claude Code workers.

Star the repo if this was useful: https://github.com/hdubey-debug/loom